# Export der Wochenmodelle

Für die Stichtage 6, 13, …, 111 wird jeweils ein eigenes Modell
gespeichert. Das Training nutzt die Präsentationen vor 2014J.

Aus der Demo mit 2014J entfernen wir Personen, die bereits in einer
früheren Präsentation vorkommen. So bleiben Training und Demo nach
Personen getrennt und die Demo entspricht unserer bisherigen
zeitlichen Prüfung.

Das Modell verwendet Kurs-, Anmelde-, Assessment- und Klickmerkmale.
`highest_education` bleibt nach dem Gruppenvergleich außerhalb des
Hinweismodells. Die Demo-CSV enthält keine späteren Kursergebnisse.
2014J wurde im Projekt bereits explorativ betrachtet und ist daher
kein unberührter Abschlusstest.

In [1]:
from pathlib import Path
import pandas as pd

# Projektordner finden, auch wenn das Notebook im Ordner "notebooks" liegt.
projektordner = Path.cwd()
if projektordner.name.lower() == "notebooks":
    projektordner = projektordner.parent

basis_datei = (
    projektordner
    / "data"
    / "processed"
    / "oulad_wochenmodellbasis_tag6_bis111.csv"
)
assert basis_datei.exists(), f"Datei nicht gefunden: {basis_datei}"

basis = pd.read_csv(basis_datei)

stichtage = list(range(6, 112, 7))
schluessel = ["code_module", "code_presentation", "id_student"]

# Diese Merkmale verwendet das Wochenmodell.
# id_student dient nur zur Zuordnung; stichtag wählt das Wochenmodell.
modellspalten = [
    "code_module",
    "code_presentation",
    "anmeldetag",
    "assessments_faellig",
    "abgaben",
    "banked_faellige",
    "assessments_fehlend",
    "klicks_bis_stichtag",
    "klicks_letzte_7_tage",
]

eingabespalten = schluessel + ["stichtag"] + modellspalten[2:]

# Prüfen, ob die aufbereitete Datei zum erwarteten Schema passt.
assert sorted(basis["stichtag"].unique().tolist()) == stichtage
assert not basis.duplicated(["stichtag"] + schluessel).any()
assert set(eingabespalten).issubset(basis.columns)
assert not basis[
    eingabespalten + ["abbruch_oder_fail"]
].isna().any().any()

# Alle Präsentationen vor 2014J bilden die Trainingsgruppe.
training = basis.loc[
    basis["code_presentation"].ne("2014J")
].copy()

# 2014J bildet die zeitliche Prüf- und Demogruppe.
pruefung = basis.loc[
    basis["code_presentation"].eq("2014J")
].copy()

# Die ältere Wochenbasis enthält auch Stichtage nach Tag 111.
# Damit finden wir dieselben zuvor eingeschriebenen Personen wie
# bei der früheren zeitlichen Prüfung.
alte_basis_datei = (
    projektordner
    / "data"
    / "processed"
    / "oulad_wochenmodellbasis.csv"
)
assert alte_basis_datei.exists(), (
    f"Ältere Wochenbasis nicht gefunden: {alte_basis_datei}"
)

alte_basis = pd.read_csv(
    alte_basis_datei,
    usecols=["code_presentation", "id_student"],
)

# Personen aus früheren Präsentationen beider Wochenbasen erfassen.
fruehere_personen = set(training["id_student"])
fruehere_personen.update(
    alte_basis.loc[
        alte_basis["code_presentation"].ne("2014J"),
        "id_student",
    ]
)

# Diese Personen aus der 2014J-Prüfgruppe entfernen.
gemeinsame_personen = fruehere_personen.intersection(
    set(pruefung["id_student"])
)
pruefung = pruefung.loc[
    ~pruefung["id_student"].isin(gemeinsame_personen)
].copy()

# Die tatsächlich für Modelle verwendeten Trainingspersonen
# dürfen in der Demo nicht vorkommen.
assert set(training["id_student"]).isdisjoint(
    set(pruefung["id_student"])
)
assert len(pruefung) > 0

print("Stichtage:", stichtage)
print("Benötigte Upload-Spalten:", eingabespalten)
print("Aus 2014J ausgeschlossene Personen:", len(gemeinsame_personen))
print("Trainings-Datenstände:", len(training))
print("Demo-Datenstände:", len(pruefung))
print(
    "Verschiedene Demo-Kurseinträge:",
    len(pruefung[schluessel].drop_duplicates()),
)

Stichtage: [6, 13, 20, 27, 34, 41, 48, 55, 62, 69, 76, 83, 90, 97, 104, 111]
Benötigte Upload-Spalten: ['code_module', 'code_presentation', 'id_student', 'stichtag', 'anmeldetag', 'assessments_faellig', 'abgaben', 'banked_faellige', 'assessments_fehlend', 'klicks_bis_stichtag', 'klicks_letzte_7_tage']
Aus 2014J ausgeschlossene Personen: 1403
Trainings-Datenstände: 282519
Demo-Datenstände: 121447
Verschiedene Demo-Kurseinträge: 8682


In [2]:
import joblib
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

modellordner = projektordner / "models" / "oulad_wochen"
modellordner.mkdir(parents=True, exist_ok=True)

kategorien = ["code_module", "code_presentation"]
zahlen = [
    spalte for spalte in modellspalten
    if spalte not in kategorien
]

for stichtag in stichtage:
    # Für jede Woche wird nur mit Datenständen derselben Woche trainiert.
    trainingswoche = training.loc[
        training["stichtag"].eq(stichtag)
    ]
    assert trainingswoche["abbruch_oder_fail"].nunique() == 2

    # Die Verarbeitung wird zusammen mit dem Modell gespeichert.
    vorbereitung = ColumnTransformer([
        (
            "kategorien",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            kategorien,
        ),
        ("zahlen", StandardScaler(), zahlen),
    ])

    modell = Pipeline([
        ("vorbereitung", vorbereitung),
        (
            "klassifikator",
            HistGradientBoostingClassifier(random_state=42),
        ),
    ])

    modell.fit(
        trainingswoche[modellspalten],
        trainingswoche["abbruch_oder_fail"].astype(int),
    )

    datei = modellordner / f"tag_{stichtag:03d}.joblib"
    joblib.dump(modell, datei)

    # Kurz prüfen, ob das gespeicherte Modell wieder geladen
    # und auf zurückgelegte 2014J-Eingaben angewendet werden kann.
    probe = pruefung.loc[
        pruefung["stichtag"].eq(stichtag),
        modellspalten,
    ].head(5)

    assert len(probe) == 5
    risiko = joblib.load(datei).predict_proba(probe)[:, 1]
    assert np.isfinite(risiko).all()

    print(
        f"Tag {stichtag}: "
        f"{len(trainingswoche)} Trainings-Datenstände "
        f"-> {datei.name}"
    )

Tag 6: 18934 Trainings-Datenstände -> tag_006.joblib
Tag 13: 18724 Trainings-Datenstände -> tag_013.joblib
Tag 20: 18583 Trainings-Datenstände -> tag_020.joblib
Tag 27: 18309 Trainings-Datenstände -> tag_027.joblib
Tag 34: 18126 Trainings-Datenstände -> tag_034.joblib
Tag 41: 17968 Trainings-Datenstände -> tag_041.joblib
Tag 48: 17786 Trainings-Datenstände -> tag_048.joblib
Tag 55: 17632 Trainings-Datenstände -> tag_055.joblib
Tag 62: 17490 Trainings-Datenstände -> tag_062.joblib
Tag 69: 17360 Trainings-Datenstände -> tag_069.joblib
Tag 76: 17251 Trainings-Datenstände -> tag_076.joblib
Tag 83: 17146 Trainings-Datenstände -> tag_083.joblib
Tag 90: 17014 Trainings-Datenstände -> tag_090.joblib
Tag 97: 16870 Trainings-Datenstände -> tag_097.joblib
Tag 104: 16719 Trainings-Datenstände -> tag_104.joblib
Tag 111: 16607 Trainings-Datenstände -> tag_111.joblib


In [3]:
import json

# Das Schema beschreibt die erwartete Upload-Datei und die Modell-Dateien.
schema = {
    "version": "wochenmodell_v1",
    "stichtage": stichtage,
    "eingabespalten": eingabespalten,
    "modellspalten": modellspalten,
    "modellpfad_muster": "tag_{stichtag:03d}.joblib",
    "hinweisanteile_je_kurs": [10, 20],
}

schema_datei = modellordner / "schema.json"
with schema_datei.open("w", encoding="utf-8") as datei:
    json.dump(schema, datei, ensure_ascii=False, indent=2)

# pruefung enthält die bereinigten Kurseinträge aus 2014J.
# demo enthält daraus ausschließlich die Spalten für den Upload.
demo = pruefung[eingabespalten].copy()

assert not demo.duplicated(["stichtag"] + schluessel).any()
assert "abbruch_oder_fail" not in demo.columns
assert "final_result" not in demo.columns
assert "highest_education" not in demo.columns
assert not demo.isna().any().any()

demo_ordner = projektordner / "data" / "demo"
demo_ordner.mkdir(parents=True, exist_ok=True)

demo_datei = demo_ordner / "oulad_wochen_2014j.csv"
demo.to_csv(demo_datei, index=False)

print("Schema:", schema_datei)
print("Demo:", demo_datei)
print("Demo-Datenstände:", len(demo))
print(
    "Verschiedene Kurseinträge:",
    len(demo[schluessel].drop_duplicates()),
)
print("Spalten:", demo.columns.tolist())

Schema: c:\Users\Tony Willert\OneDrive\Weiterbildung\PortfolioProjekt\models\oulad_wochen\schema.json
Demo: c:\Users\Tony Willert\OneDrive\Weiterbildung\PortfolioProjekt\data\demo\oulad_wochen_2014j.csv
Demo-Datenstände: 121447
Verschiedene Kurseinträge: 8682
Spalten: ['code_module', 'code_presentation', 'id_student', 'stichtag', 'anmeldetag', 'assessments_faellig', 'abgaben', 'banked_faellige', 'assessments_fehlend', 'klicks_bis_stichtag', 'klicks_letzte_7_tage']
